# Lecture 4: Downloading Historical FMI Weather Station Data

This notebook demonstrates how to discover and download historical weather observations from the **Finnish Meteorological Institute (FMI) Open Data WFS API** (`https://opendata.fmi.fi/wfs`, documented at [en.ilmatieteenlaitos.fi/open-data-manual](https://en.ilmatieteenlaitos.fi/open-data-manual)) in Google Colab.

## 1. Investigation: Available FMI Historical Data for Farm Analytics (2020–2026)

FMI publishes continuously updated station observations via OGC **Web Feature Service (WFS 2.0.0)** stored queries. For a farm **milk robot** and/or **tractor telemetry** dataset, the most relevant historical observation queries are:

| Stored Query ID | Resolution | Key Variables | Max Window / Request | Farm Relevance |
|---|---|---|---|---|
| `fmi::observations::weather::hourly::simple` | Hourly | `TA_PT1H_AVG` (°C), `RH_PT1H_AVG` (%), `PRA_PT1H_ACC` (mm), `WS_PT1H_AVG` (m/s), `PA_PT1H_AVG` (hPa) | 31 days | **Milk robot:** Temperature–Humidity Index (THI) for dairy cow heat stress, barn ventilation, pressure fronts. **Tractor:** Hourly rain delays, spraying wind limits. |
| `fmi::observations::weather::daily::simple` | Daily | `tday`, `tmin`, `tmax` (°C), `rrday` (mm), `snow` (cm) | 365 days | **Tractor:** Field trafficability (wetness from `rrday`), frost/thaw timing (`tmin`), winter snow depth (`snow`). **Milk robot:** Daily heat load & seasonal trends. |
| `fmi::observations::radiation::simple` | 1 min (supports `timestep=60`) | `GLOB_1MIN` (W/m²), `SUND_1MIN` (s) | 7 days | **Tractor:** Crop drying / evapotranspiration, daylight conditions. **Milk robot:** Solar heat load on barn/pasture. |
| `fmi::ef::stations` | Metadata | `fmisid`, station name, latitude, longitude | All stations | Used to rank stations by Haversine distance from the farm. |

> **Note on soil observations:** Although `fmi::observations::soil::hourly::simple` is listed in FMI's stored queries, it currently returns no active observations for these stations. Daily/hourly precipitation (`rrday`, `PRA_PT1H_ACC`), temperature (`tday`, `tmin`), and snow depth (`snow`) therefore serve as practical proxies for soil moisture and field trafficability.

## 2. Per-Variable Nearest-Station Strategy

Not every FMI weather station measures every variable (for example, some stations only measure daily precipitation and snow depth, while others lack wind or air pressure). This notebook:
1. Ranks all FMI stations by distance from the target coordinates (default: **60.817° N, 23.767° E** — Mustiala).
2. Probes stations in distance order over a short sample window to find, **for each requested variable independently**, the nearest station that actually reports valid data for that variable.
3. Downloads the full historical range (default **2020-01-01 to latest**) in API-safe time chunks, grouping variables that share the same station and stored query into a single request.

In [ ]:
import math
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd

# --- Configuration ---
TARGET_LAT = 60.817  # Default: Mustiala (60.817 N, 23.767 E)
TARGET_LON = 23.767

START_DATE = "2020-01-01"
# Defaults to today's UTC date so it always fetches up to the latest available data
END_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Set to False if you only need daily data (daily takes about 4s; hourly 2020-2026 takes about 45s)
DOWNLOAD_HOURLY = True
DOWNLOAD_DAILY = True

print(f"Target location: {TARGET_LAT:.3f} N, {TARGET_LON:.3f} E")
print(f"Date range: {START_DATE} to {END_DATE}")

In [ ]:
# Variable specifications: (stored_query_id, max_days_per_chunk, friendly_column_name, description)
DAILY_VARIABLES = {
    "tday": ("fmi::observations::weather::daily::simple", 365, "daily_temp_mean_c", "Daily mean temp (°C)"),
    "tmin": ("fmi::observations::weather::daily::simple", 365, "daily_temp_min_c", "Daily min temp (°C)"),
    "tmax": ("fmi::observations::weather::daily::simple", 365, "daily_temp_max_c", "Daily max temp (°C)"),
    "rrday": ("fmi::observations::weather::daily::simple", 365, "daily_precip_mm", "Daily precip (mm)"),
    "snow": ("fmi::observations::weather::daily::simple", 365, "snow_depth_cm", "Snow depth (cm)"),
}

HOURLY_VARIABLES = {
    "TA_PT1H_AVG": ("fmi::observations::weather::hourly::simple", 30, "temp_avg_c", "Hourly temp (°C)"),
    "RH_PT1H_AVG": ("fmi::observations::weather::hourly::simple", 30, "humidity_pct", "Relative humidity (%)"),
    "PRA_PT1H_ACC": ("fmi::observations::weather::hourly::simple", 30, "precip_mm", "Hourly precip (mm)"),
    "WS_PT1H_AVG": ("fmi::observations::weather::hourly::simple", 30, "wind_speed_ms", "Wind speed (m/s)"),
    "PA_PT1H_AVG": ("fmi::observations::weather::hourly::simple", 30, "pressure_hpa", "Air pressure (hPa)"),
}

In [ ]:
FMI_WFS_URL = "https://opendata.fmi.fi/wfs"
NS = {
    "ef": "http://inspire.ec.europa.eu/schemas/ef/4.0",
    "gml": "http://www.opengis.net/gml/3.2",
    "BsWfs": "http://xml.fmi.fi/schema/wfs/2.0",
}


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two (lat, lon) points in kilometers."""
    earth_radius_km = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * earth_radius_km * math.asin(math.sqrt(a))


def fetch_xml(params, retries=3, timeout=30):
    """Fetch and parse XML from FMI WFS with simple retry/backoff."""
    url = FMI_WFS_URL + "?" + urllib.parse.urlencode(params)
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as resp:
                return ET.fromstring(resp.read())
        except urllib.error.HTTPError as err:
            if err.code == 400:
                raise  # Invalid query/station combination; do not retry
            if attempt == retries - 1:
                raise
        except Exception:
            if attempt == retries - 1:
                raise
        time.sleep(1.5 * (attempt + 1))


def get_stations_sorted(lat, lon):
    """Return all FMI stations sorted by distance (km) from (lat, lon)."""
    root = fetch_xml({
        "service": "WFS",
        "version": "2.0.0",
        "request": "getFeature",
        "storedquery_id": "fmi::ef::stations",
    })
    rows = []
    for fac in root.findall(".//ef:EnvironmentalMonitoringFacility", NS):
        fmisid = fac.findtext("gml:identifier", default="", namespaces=NS)
        name = fac.findtext("ef:name", default="", namespaces=NS)
        pos = fac.findtext(".//gml:pos", default="", namespaces=NS).strip().split()
        if len(pos) == 2:
            slat, slon = float(pos[0]), float(pos[1])
            rows.append({
                "fmisid": fmisid,
                "name": name,
                "lat": slat,
                "lon": slon,
                "distance_km": haversine_km(lat, lon, slat, slon),
            })
    return pd.DataFrame(rows).sort_values("distance_km").reset_index(drop=True)


def find_nearest_stations_per_variable(stations_df, variable_specs, probe_end_date, max_stations_to_check=25):
    """
    For each variable in variable_specs, find the nearest station that returns
    non-NaN observations during a 3-day probe window ending on probe_end_date.
    """
    end_dt = datetime.strptime(probe_end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    start_dt = end_dt - timedelta(days=3)
    probe_start = start_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    probe_end = end_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    # Group variables by stored_query_id
    by_query = defaultdict(list)
    for param, spec in variable_specs.items():
        by_query[spec[0]].append(param)

    assignments = {}
    for stored_query, params in by_query.items():
        remaining = set(params)
        for _, st in stations_df.head(max_stations_to_check).iterrows():
            if not remaining:
                break
            try:
                root = fetch_xml({
                    "service": "WFS",
                    "version": "2.0.0",
                    "request": "getFeature",
                    "storedquery_id": stored_query,
                    "fmisid": st["fmisid"],
                    "parameters": ",".join(sorted(remaining)),
                    "starttime": probe_start,
                    "endtime": probe_end,
                })
            except urllib.error.HTTPError:
                continue

            valid_params = {
                el.findtext("BsWfs:ParameterName", default="", namespaces=NS)
                for el in root.findall(".//BsWfs:BsWfsElement", NS)
                if el.findtext("BsWfs:ParameterValue", default="", namespaces=NS).lower() not in ("", "nan")
            }

            for p in sorted(valid_params & remaining):
                _, chunk_days, col_name, desc = variable_specs[p]
                assignments[p] = {
                    "fmi_param": p,
                    "column": col_name,
                    "description": desc,
                    "stored_query": stored_query,
                    "chunk_days": chunk_days,
                    "fmisid": st["fmisid"],
                    "station_name": st["name"],
                    "distance_km": round(st["distance_km"], 2),
                }
                remaining.remove(p)

        if remaining:
            print(f"Warning: No station found within nearest {max_stations_to_check} for {sorted(remaining)}")

    return pd.DataFrame([assignments[p] for p in variable_specs if p in assignments])


def download_assigned_variables(assignments_df, start_date, end_date):
    """
    Download all assigned variables in [start_date, end_date] using chunked WFS calls.
    Variables sharing the same (stored_query, fmisid) are fetched together.
    """
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end_dt = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)

    records = defaultdict(dict)
    grouped = assignments_df.groupby(["stored_query", "fmisid", "station_name", "chunk_days"])

    for (stored_query, fmisid, station_name, chunk_days), group in grouped:
        param_to_col = dict(zip(group["fmi_param"], group["column"]))
        param_str = ",".join(sorted(param_to_col.keys()))
        print(f"Downloading {list(param_to_col.values())} from {station_name} (FMISID {fmisid})...")

        cur = start_dt
        while cur < end_dt:
            nxt = min(cur + timedelta(days=int(chunk_days)), end_dt)
            root = fetch_xml({
                "service": "WFS",
                "version": "2.0.0",
                "request": "getFeature",
                "storedquery_id": stored_query,
                "fmisid": fmisid,
                "parameters": param_str,
                "starttime": cur.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "endtime": nxt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
            for el in root.findall(".//BsWfs:BsWfsElement", NS):
                ts = el.findtext("BsWfs:Time", default="", namespaces=NS)
                p = el.findtext("BsWfs:ParameterName", default="", namespaces=NS)
                v = el.findtext("BsWfs:ParameterValue", default="", namespaces=NS)
                if ts and p in param_to_col:
                    val = float(v) if v and v.lower() != "nan" else float("nan")
                    records[ts][param_to_col[p]] = val
            cur = nxt

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame.from_dict(records, orient="index")
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "timestamp_utc"
    ordered_cols = [c for c in assignments_df["column"] if c in df.columns]
    return df[ordered_cols].sort_index()

## 3. Station Discovery & Per-Variable Assignment

Let's list the 10 nearest FMI stations to `(TARGET_LAT, TARGET_LON)` and match each daily and hourly variable to the nearest station that provides it.

In [ ]:
stations_df = get_stations_sorted(TARGET_LAT, TARGET_LON)
print(f"Found {len(stations_df)} FMI stations. Nearest 10 to ({TARGET_LAT}, {TARGET_LON}):")
display(stations_df.head(10))

daily_assignments = find_nearest_stations_per_variable(stations_df, DAILY_VARIABLES, END_DATE)
hourly_assignments = find_nearest_stations_per_variable(stations_df, HOURLY_VARIABLES, END_DATE)

print("\nAssigned station per daily variable:")
display(daily_assignments[["fmi_param", "column", "station_name", "fmisid", "distance_km"]])

print("\nAssigned station per hourly variable:")
display(hourly_assignments[["fmi_param", "column", "station_name", "fmisid", "distance_km"]])

### Optional Check: Per-Variable Fallback in Action

For Mustiala (`60.817 N, 23.767 E`), the nearest station (**Jokioinen Ilmala**, approx. 14.6 km away) is a major observatory that measures all listed variables. If you instead pick a location near a smaller station — such as **Hattula Lepaa** (`61.10 N, 24.30 E`), which measures temperature, relative humidity, and precipitation but **not** wind speed or air pressure — the code automatically falls back to **Hämeenlinna Katinen** (approx. 15.2 km away) for wind and pressure. Run the cell below to verify this behavior.

In [ ]:
demo_stations = get_stations_sorted(61.10, 24.30)
demo_assignments = find_nearest_stations_per_variable(demo_stations, HOURLY_VARIABLES, END_DATE)
display(demo_assignments[["fmi_param", "column", "station_name", "fmisid", "distance_km"]])

## 4. Download Historical Weather Data (2020–2026)

In [ ]:
t0 = time.time()

if DOWNLOAD_DAILY:
    df_daily = download_assigned_variables(daily_assignments, START_DATE, END_DATE)
    print(f"\nDaily DataFrame shape: {df_daily.shape}")
    display(df_daily.head())

if DOWNLOAD_HOURLY:
    df_hourly = download_assigned_variables(hourly_assignments, START_DATE, END_DATE)
    print(f"\nHourly DataFrame shape: {df_hourly.shape}")
    display(df_hourly.head())

print(f"\nTotal download time: {time.time() - t0:.1f} s")

## 5. Plot Raw Weather Variables

Plot the raw downloaded daily and hourly variables directly before merging (by timestamp) with milk robot or tractor telemetry data.

In [ ]:
import textwrap

if DOWNLOAD_DAILY and not df_daily.empty:
    daily_labels = dict(zip(daily_assignments["column"], daily_assignments["description"]))
    axes = df_daily.plot(subplots=True, figsize=(12, 10), title="Raw Daily FMI Observations")
    for ax, col in zip(axes, df_daily.columns):
        ax.set_ylabel(textwrap.fill(daily_labels.get(col, col), width=14))
    axes[-1].set_xlabel("Timestamp (UTC)")
    plt.tight_layout()
    plt.show()

if DOWNLOAD_HOURLY and not df_hourly.empty:
    hourly_labels = dict(zip(hourly_assignments["column"], hourly_assignments["description"]))
    axes = df_hourly.plot(subplots=True, figsize=(12, 10), title="Raw Hourly FMI Observations")
    for ax, col in zip(axes, df_hourly.columns):
        ax.set_ylabel(textwrap.fill(hourly_labels.get(col, col), width=14))
    axes[-1].set_xlabel("Timestamp (UTC)")
    plt.tight_layout()
    plt.show()

## 6. Save to CSV (Optional)

Save the downloaded tables to CSV so you can merge them with your milk robot or tractor telemetry timestamps.

In [ ]:
if DOWNLOAD_DAILY and not df_daily.empty:
    df_daily.to_csv("fmi_weather_daily_2020_2026.csv")
    print("Saved fmi_weather_daily_2020_2026.csv")

if DOWNLOAD_HOURLY and not df_hourly.empty:
    df_hourly.to_csv("fmi_weather_hourly_2020_2026.csv")
    print("Saved fmi_weather_hourly_2020_2026.csv")